# Example: Using the C19EpidemiologyGraphs database to create a Torch Geometric dataset for Graph Neural Network (GNN) usage

This example notebook serves to show an use case of the C19EpidemiologyGraphs database. Specifically, we are creating a Torch Geometric dataset, which is a frequent step before a Graph Neural Network can be trained. Detailed explanations regarding Torch Geometric's dataset creation process are beyond our scope, but we refer the curious reader to the official tutorial on the matter: https://pytorch-geometric.readthedocs.io/en/latest/tutorial/create_dataset.html.
It's worth mentioning that the linked tutorial tackles the creation of in-memory datasets, and here we are going to use an on-disk dataset, where the entries are stored in an SQLite database (a completely different one than C19EpidemiologyGraphs).

With regards to the epidemiology table, we are going to use the interpolated columns rather than only relying on the true values, and we are also going to be referencing confirmed cases, deaths and testing, but NOT recoveries. We're also going to perform several filters, specifically:
- The rolling time window sizes must be between 1 (exclusive) and 30 (inclusive).
- To try and enforce data quality, we are only going to allow epidemiology rows whose time windows have, for each of the relevant columns, at least 90% of it's days present (interpolated or not), and 80% of it's days present specifically with non-interpolated/real values.
- All relevant columns must be non-negative. Some corrections of previous data can result in negative values, but for our purposes that's not desirable, so those rows are removed.
- The number of tests must be higher than or equal to the number of confirmed cases.

The filtered epidemiology table is then loaded into memory with Polars and grouped by the location's aggregation level, window starting date, window size and, sometimes, country name. The country name is only added as a group column if, and only if, aggregation level is 2. Then, for each group, we obtain the graph nodes and edges, based on whether the locations involved have epidemiological information in the current group. This creates a graph, which is then broken into it's connected components. Lastly, only components with size (measured as the number of nodes) larger than or equal to 8 get added to the dataset.

Different use cases would warrant different filters, and this is only showing one possible set of restrictions and transformations. All of this is to show the flexibility of C19EpidemiologyGraphs.

In [31]:
from typing import Optional, Callable, Any
import datetime
from tqdm import tqdm
import polars as pl
import torch
import torch_geometric as tg

In [35]:
class ExampleDatasetClass(tg.data.OnDiskDataset):
    def __init__(
            self,
            root: str,
            min_graph_size: int = 8,
            min_non_absent: float = 0.9,
            min_non_interpolated: float = 0.8,
            transform: Optional[Callable] = None,
            pre_transform: Optional[Callable] = None,
            pre_filter: Optional[Callable] = None,
            backend: str = "sqlite",
            # Minimum number of torch geometric graphs accumulated in memory before a SQLite insert.
            block_size: int = 4096,
            log: bool = True
    ) -> None:
        self.min_graph_size = min_graph_size
        self.min_non_absent = min_non_absent
        self.min_non_interpolated = min_non_interpolated
        self.pre_transform = pre_transform
        self.block_size = block_size
        schema = {
            "x": dict(dtype=torch.int32, size=(-1, 1)),
            "edge_index": dict(dtype=torch.int64, size=(2, -1)),
            "edge_attr": dict(dtype=torch.float32, size=(-1, 1)),
            "pos": dict(dtype=torch.float32, size=(-1, 2)),
            "window_epidemiology": dict(dtype=torch.int32, size=(-1, 3)),
            "cumulative_epidemiology": dict(dtype=torch.int64, size=(-1, 3)),
            "aggregation_level": int,
            "window_starting_date": int,
            "window_size": int,
            "grouped_by_country": str,
            "component_index": int
        }
        super().__init__(root, transform, pre_filter, backend, schema, log)

    @property
    def raw_file_names(self):
        return ["C19EpidemiologyGraphs.sqlite3"]

    def process_and_extend(self, data_list):
        if self.pre_filter is not None:
            data_list = filter(self.pre_filter, data_list)
        if self.pre_transform is not None:
            data_list = map(self.pre_transform, data_list)
        if self.pre_filter is not None or self.pre_transform is not None:
            data_list = list(data_list)
        if len(data_list) > 0:
            self.extend(data_list)

    def process(self):
        group_columns = ["aggregation_level", "window_starting_date", "window_size", "grouped_by_country"]
        uri = "sqlite://" + self.root + "/" + self.raw_file_names[0]
        # Several columns from epidemiology and vertices are going to be ignored in this example,
        #   such as all metrics related to recovered patients.
        query_valid_epidemiology = """
            SELECT v.aggregation_level, e.window_starting_date, e.window_size, e.location_key,
                   e.interpolated_new_confirmed, e.interpolated_new_deceased, e.interpolated_new_tested,
                   e.cumulative_confirmed, e.cumulative_deceased, e.cumulative_tested,
                   v.ROWID AS node_rowid, v.x_longitude, v.y_latitude,
                   CASE
                       WHEN v.aggregation_level = 2 THEN v.country_name
                       ELSE ''
                   END AS grouped_by_country
            FROM epidemiology AS e
            INNER JOIN vertices AS v ON e.location_key = v.location_key
            WHERE e.window_size > 1
              AND e.window_size <= 30
              AND e.num_non_nulls_in_true_confirmed / e.window_size >= {0}
              AND e.num_non_nulls_in_true_deceased / e.window_size >= {0}
              AND e.num_non_nulls_in_true_tested / e.window_size >= {0}
              AND e.num_non_nulls_in_interpolated_confirmed / e.window_size >= {1}
              AND e.num_non_nulls_in_interpolated_deceased / e.window_size >= {1}
              AND e.num_non_nulls_in_interpolated_tested / e.window_size >= {1}
              AND e.interpolated_new_confirmed >= 0
              AND e.interpolated_new_deceased >= 0
              AND e.interpolated_new_tested >= 0
              AND e.interpolated_new_tested >= e.interpolated_new_confirmed
        """.format(self.min_non_interpolated, self.min_non_absent)
        valid_epidemiology = pl.read_database_uri(query=query_valid_epidemiology, uri=uri, engine="connectorx",
                                                  partition_on="window_starting_date",
                                                  partition_num=10, partition_range=(18261, 19356))
        epidemiology_groups = valid_epidemiology.group_by(group_columns)
        num_epidemiology_groups = valid_epidemiology.n_unique(subset=group_columns)
        data_list = []
        for ((aggregation_level, window_starting_date, window_size, grouped_by_country),
             group) in tqdm(epidemiology_groups, total=num_epidemiology_groups):
            group_location_keys = group.get_column("location_key").to_list()
            nodes = group.select("node_rowid", "x_longitude", "y_latitude",
                                 "interpolated_new_confirmed", "interpolated_new_deceased", "interpolated_new_tested",
                                 "cumulative_confirmed", "cumulative_deceased", "cumulative_tested")
            # A graph's edges are stored as indices relative to the position of the nodes in graph.x, so we need
            #   to translate the fixed and unique location_key/rowid (in this case, rowid) of a vertex into a relative
            #   index that's starting from 0 and is monotonically increasing with every subsequent node.
            # This is done by using polars' DataFrame.with_row_index() to create a 1-to-1 mapping that can then
            #   be joined with the edges' source nodes, and again with the destination nodes.
            nodes_map = nodes.with_row_index().select("node_rowid", "index")
            q_edges = """
                WITH curr_nodes AS (SELECT location_key, ROWID AS node_rowid
                                    FROM vertices
                                    WHERE location_key IN ({0}))
                SELECT source_verts.node_rowid AS source_id,
                       destination_verts.node_rowid AS destination_id,
                       edges.distance_m
                FROM edges
                INNER JOIN curr_nodes AS source_verts ON edges.source = source_verts.location_key
                INNER JOIN curr_nodes AS destination_verts ON edges.destination = destination_verts.location_key
            """.format(', '.join(['?'] * len(group)))
            edges = pl.read_database_uri(query=q_edges, uri=uri, engine="adbc",
                                         execute_options={"parameters": tuple(group_location_keys)}).join(
                nodes_map, how="inner", left_on="source_id", right_on="node_rowid", maintain_order="left"
            ).join(
                nodes_map, how="inner", left_on="destination_id", right_on="node_rowid",
                suffix="_destination", maintain_order="left"
            )
            # With all the required information related to the graph and it's nodes and edges collected,
            #   it's possible to instantiate a Torch Geometric graph.
            # We need to convert Polars' DataFrames to Torch tensors,
            #   but 0-dimensional ints and strings can remain as primitives.
            graph = tg.data.Data(
                # The x feature is mandatory in Torch Geometric. Here, we used the node's unique rowid in the database
                x=nodes.select("node_rowid").to_torch(dtype=pl.Int32),
                edge_index=edges.select("index", "index_destination").to_torch(dtype=pl.Int64).T,
                edge_attr=edges.select("distance_m").to_torch(dtype=pl.Float32),
                pos=nodes.select("x_longitude", "y_latitude").to_torch(dtype=pl.Float32),
                window_epidemiology=nodes.select("interpolated_new_confirmed", "interpolated_new_deceased",
                                                 "interpolated_new_tested").to_torch(dtype=pl.Int32),
                cumulative_epidemiology=nodes.select("cumulative_confirmed", "cumulative_deceased",
                                                     "cumulative_tested").to_torch(dtype=pl.Int64),
                aggregation_level=aggregation_level,
                # Like in C19EpidemiologyGraphs, dates in this dataset's SQLite database are stored as the
                #   number of days since the Unix epoch (01/01/1970).
                #   However, when the database entries are materialized as actual Torch Geometric graphs
                #   (like here, before this graph is written into the database),
                #   it's possible to use datetime objects, to better represent these dates.
                # The conversion to and from datetime and the number of days since the Unix epoch can be seen
                #   in the methods serialize and deserialize, a bit under.
                window_starting_date=datetime.date(1970, 1, 1) + datetime.timedelta(days=window_starting_date),
                window_size=window_size,
                grouped_by_country=grouped_by_country
            ).coalesce()
            # For this example, we're not adding this full graph, but rather it's connected components.
            #   Torch Geometric's Data objects have a method for this.
            # We also take this moment to filter these components based on their size, measured as number of nodes.
            graph_components = list(filter(lambda data: data.num_nodes >= self.min_graph_size,
                                           graph.connected_components()))
            for i in range(len(graph_components)):
                graph_components[i].component_index = i
            data_list.extend(graph_components)
            # Note that the length of data_list usually exceeds block_size a bit,
            #   as this check is outside the group for-loop, therefore happening only once per
            #   C19EpidemiologyGraphs' graph entry, which can include many entries into the dataset.
            # No reason in particular for this to be done like that,
            #   just a very small memory-performance tradeoff decision.
            if len(data_list) >= self.block_size:
                self.process_and_extend(data_list)
                data_list = []
        self.process_and_extend(data_list)

    # Convert the window_starting_date from a datetime object to the number of days since the Unix epoch.
    def serialize(self, data: tg.data.data.BaseData) -> Any:
        d = data.to_dict()
        d["window_starting_date"] = (d["window_starting_date"] - datetime.date(1970, 1, 1)).days
        return d

    # Convert the window_starting_date from the number of days since the Unix epoch to a datetime object.
    def deserialize(self, data: Any) -> tg.data.data.BaseData:
        data["window_starting_date"] = datetime.date(1970, 1, 1) + datetime.timedelta(days=data["window_starting_date"])
        return tg.data.Data.from_dict(data)

In [36]:
ds = ExampleDatasetClass(".", min_graph_size=8, min_non_absent=0.9, min_non_interpolated=0.8)
ds

Processing...
100%|██████████| 31057/31057 [20:36<00:00, 25.11it/s]  
Done!


ExampleDatasetClass(179272)

In [37]:
ds[100]

Data(x=[484, 1], edge_index=[2, 2314], edge_attr=[2314, 1], pos=[484, 2], window_epidemiology=[484, 3], cumulative_epidemiology=[484, 3], aggregation_level=2, window_starting_date=2021-03-23, window_size=3, grouped_by_country='Brazil', component_index=15)

In [39]:
ds.get_summary()

100%|██████████| 179272/179272 [00:05<00:00, 31173.79it/s]


ExampleDatasetClass (#graphs=179272):
+------------+----------+----------+
|            |   #nodes |   #edges |
|------------+----------+----------|
| mean       |     33.4 |    157.1 |
| std        |     81.5 |    409.5 |
| min        |      8   |     22   |
| quantile25 |     10   |     39   |
| median     |     16   |     63   |
| quantile75 |     34   |    170   |
| max        |   3065   |  16795   |
+------------+----------+----------+

And it's done! A Torch Geometric dataset with over 179K entries and with medians of 16 nodes and 63 edges (bear in mind that as these are undirected graphs whose edges are stored as directed, unless an edge is a self loop, then it's counted twice). This dataset can now be incorporated into a DataLoader, transformed/pre-processed and used for GNN training and/or inference. We won't show it being actually used this way here, as it's not unique at all to our work, but we hope this example use case illustrated how to bridge any gap between C19EpidemiologyGraphs and third-party libraries.

Oh, and always remember to close your database connection when done using them! ^^

In [40]:
ds.close()